In [ ]:
import os
import json
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset, Dataset
from torchvision import datasets, transforms, models
from sklearn.model_selection import train_test_split
from tqdm import tqdm

In [16]:

# FIX: Define TransformWrapper at top-level so pickle can serialize it across workers
class TransformWrapper(torch.utils.data.Dataset):
    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform

    def __getitem__(self, idx):
        img, label = self.subset[idx]
        return self.transform(img), label

    def __len__(self):
        return len(self.subset)


def main():
    # 1. Hardware & Dataset Paths
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Executing training on: {device}")

    DATA_DIR = r"C:\Users\harsh\OneDrive\Desktop\plant class\without background class"
    SAVE_MODEL_PATH = "medicinal_plant_model.pth"
    CLASSES_PATH = "classes.json"

    IMG_SIZE = (384, 384)
    BATCH_SIZE = 16
    EPOCHS = 25
    LEARNING_RATE = 3e-4

    # 2. Plant-Specific Data Augmentations
    train_transforms = transforms.Compose([
        transforms.Resize(IMG_SIZE),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.5),
        transforms.RandomRotation(degrees=45),
        transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    val_transforms = transforms.Compose([
        transforms.Resize(IMG_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    # 3. Stratified Dataset Loading
    base_dataset = datasets.ImageFolder(root=DATA_DIR)
    class_names = base_dataset.classes
    NUM_CLASSES = len(class_names)
    print(f"Detected {NUM_CLASSES} classes: {class_names}\n")

    with open(CLASSES_PATH, "w") as f:
        json.dump(class_names, f)

    targets = [label for _, label in base_dataset.samples]
    train_idx, val_idx = train_test_split(
        list(range(len(targets))),
        test_size=0.20,
        stratify=targets,
        random_state=42
    )

    # In Jupyter Notebook on Windows, num_workers=0 avoids pickling overhead and fork errors
    train_loader = DataLoader(
        TransformWrapper(Subset(base_dataset, train_idx), train_transforms),
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=0,
        pin_memory=True if torch.cuda.is_available() else False
    )

    val_loader = DataLoader(
        TransformWrapper(Subset(base_dataset, val_idx), val_transforms),
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        pin_memory=True if torch.cuda.is_available() else False
    )

    # 4. Multi-Layer Feature Extractor (EfficientNetV2-S Backbone)
    model = models.efficientnet_v2_s(weights=models.EfficientNet_V2_S_Weights.DEFAULT)

    # Freeze early layers
    for param in model.features[:5].parameters():
        param.requires_grad = False

    # Multi-layer classification head
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.3, inplace=True),
        nn.Linear(in_features, 512),
        nn.BatchNorm1d(512),
        nn.SiLU(),
        nn.Dropout(p=0.4),
        nn.Linear(512, NUM_CLASSES)
    )

    model = model.to(device)

    # 5. Optimization & Learning Rate Scheduling
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=LEARNING_RATE,
        weight_decay=1e-2
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

    # 6. Training & Validation Loop
    best_val_acc = 0.0

    print("Starting Training...\n" + "=" * 70)

    for epoch in range(EPOCHS):
        model.train()
        running_loss, train_correct, train_total = 0.0, 0, 0

        train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1:02d}/{EPOCHS} [Train]", leave=False)
        for images, labels in train_bar:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)
            _, preds = outputs.max(1)
            train_correct += preds.eq(labels).sum().item()
            train_total += labels.size(0)

            train_bar.set_postfix({
                "loss": f"{running_loss / train_total:.4f}",
                "acc": f"{(train_correct / train_total) * 100:.2f}%"
            })

        scheduler.step()
        epoch_train_loss = running_loss / train_total
        epoch_train_acc = (train_correct / train_total) * 100

        # Validation phase
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0

        val_bar = tqdm(val_loader, desc=f"Epoch {epoch+1:02d}/{EPOCHS} [Val]  ", leave=False)
        with torch.no_grad():
            for images, labels in val_bar:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)

                val_loss += loss.item() * images.size(0)
                _, preds = outputs.max(1)
                val_correct += preds.eq(labels).sum().item()
                val_total += labels.size(0)

                val_bar.set_postfix({
                    "val_loss": f"{val_loss / val_total:.4f}",
                    "val_acc": f"{(val_correct / val_total) * 100:.2f}%"
                })

        epoch_val_loss = val_loss / val_total
        epoch_val_acc = (val_correct / val_total) * 100

        # Epoch Summary
        print(f"Epoch [{epoch+1:02d}/{EPOCHS}] -> "
              f"Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc:.2f}% | "
              f"Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc:.2f}%")

        if epoch_val_acc > best_val_acc:
            best_val_acc = epoch_val_acc
            torch.save(model.state_dict(), SAVE_MODEL_PATH)
            print(f"  >>> Best model saved! (New peak Val Acc: {best_val_acc:.2f}%)")

        print("-" * 70)

    print(f"\nTraining Complete. Peak Validation Accuracy: {best_val_acc:.2f}%")
    print(f"Weights saved to: {SAVE_MODEL_PATH}")


if __name__ == '__main__':
    main()

Executing training on: cuda
Detected 27 classes: ['Amrathaballi', 'Arive Dantu', 'Ashoka', 'Asthma Weed', 'Balloon Vine', 'Betel', 'Bhrami', 'Bringaraja', 'Caricature', 'Catharanthus', 'Curry', 'Drumstick', 'Globe Amarnath', 'Hibiscus', 'Insulin', 'Kamakasturi', 'Karanda', 'Kohlrabi', 'Malabar Nut', 'Malabar Spinach', 'Mint', 'Nelavembu', 'Nooni', 'Padri', 'Parijatha', 'Thumbai', 'Tulsi']

Starting Training...


Epoch [01/25] -> Train Loss: 1.8489 | Train Acc: 61.05% | Val Loss: 1.0018 | Val Acc: 89.01%
  >>> Best model saved! (New peak Val Acc: 89.01%)
----------------------------------------------------------------------


Epoch [02/25] -> Train Loss: 1.0358 | Train Acc: 88.25% | Val Loss: 0.8326 | Val Acc: 95.20%
  >>> Best model saved! (New peak Val Acc: 95.20%)
----------------------------------------------------------------------


Epoch [03/25] -> Train Loss: 0.9057 | Train Acc: 92.72% | Val Loss: 0.7848 | Val Acc: 95.44%
  >>> Best model saved! (New peak Val Acc: 95.44%)
----------------------------------------------------------------------


Epoch [04/25] -> Train Loss: 0.8447 | Train Acc: 94.88% | Val Loss: 0.7965 | Val Acc: 96.02%
  >>> Best model saved! (New peak Val Acc: 96.02%)
----------------------------------------------------------------------


...

Epoch [21/25] -> Train Loss: 0.6569 | Train Acc: 99.85% | Val Loss: 0.6558 | Val Acc: 99.30%
----------------------------------------------------------------------


Epoch [22/25] -> Train Loss: 0.6564 | Train Acc: 99.88% | Val Loss: 0.6532 | Val Acc: 99.42%
----------------------------------------------------------------------


Epoch [23/25] -> Train Loss: 0.6561 | Train Acc: 99.80% | Val Loss: 0.6547 | Val Acc: 99.30%
----------------------------------------------------------------------


Epoch [24/25] -> Train Loss: 0.6552 | Train Acc: 99.88% | Val Loss: 0.6545 | Val Acc: 99.30%
----------------------------------------------------------------------


Epoch [25/25] -> Train Loss: 0.6553 | Train Acc: 99.85% | Val Loss: 0.6546 | Val Acc: 99.42%
----------------------------------------------------------------------

Training Complete. Peak Validation Accuracy: 99.53%
Weights saved to: medicinal_plant_model.pth
